# Tuning: num_leaves × learning_rate

Same job, different hyperparameters — each `fit` is an independent training
job on moto, and the `metrics.json` every run writes makes the sweep
comparable.

In [ ]:
import itertools
import json

import boto3
import pandas as pd
from sagemaker.estimator import Estimator
from sagemaker.local import LocalSession

session = LocalSession()
BUCKET = "california-house-pricing"
s3 = boto3.client(
    "s3", endpoint_url="http://california-house-pricing-moto:5000"
)
grid = list(itertools.product([31, 63, 127], [0.03, 0.05, 0.1]))
results = []
for leaves, lr in grid[:4]:
    est = Estimator(
        image_uri="california-house-pricing:train",
        role="arn:aws:iam::111111111111:role/service-role/SageMakerRole",
        instance_count=1,
        instance_type="local",
        output_path=f"s3://{BUCKET}/output",
        sagemaker_session=session,
        hyperparameters={
            "n_estimators": 300,
            "learning_rate": lr,
            "num_leaves": leaves,
        },
    )
    est.set_entry_point(
        source_dir="src/california_house_pricing", entry_point="train.py"
    )
    est.fit({"train": f"s3://{BUCKET}/input/train"})
    m = json.loads(
        s3.get_object(
            Bucket=BUCKET,
            Key=f"output/{est.latest_training_job.name}/output/metrics.json",
        )["Body"].read()
    )
    results.append(
        {
            "num_leaves": leaves,
            "learning_rate": lr,
            **m,
            "job": est.latest_training_job.name,
        }
    )
pd.DataFrame(results).sort_values("mae")

The sweep is small on purpose — the point is that hyperparameters flow
through `SM_HPS` and every job is scored the same way. Best row wins the
deploy slot.